# 02 â€” Write your first C-1N policy

The setup and function stub are supplied. You write the implementation from the assignment below, then we review your complete attempt.

Our policy will return **18 joint-target offsets**, one per actuator. Each offset is in radians relative to the neutral stance. Zero means keep that actuator's neutral target.

For the first version, the policy can ignore `observation`. We will add behavior after the function can return an action.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "learning_env.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start the kernel in spider/ or spider/notebooks/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from learning_env import LearningSimulation

sim = LearningSimulation()
measured = sim.reset()
print("Number of actuator targets:", sim.model.nu)

## Assignment: implement a neutral policy

**Objective:** write a baseline policy that commands the neutral stance.

**Interface:** `policy(observation)` returns a NumPy array of 18 floating-point joint-target offsets, in actuator order. This first version can ignore its observation input. The action length comes from the robot's actuators, not the size or type of the observation.

1. Create a fresh array named `offsets` containing 18 zeros inside the function.
2. Return that array. Zero offsets mean neutral joint targets, not zero physical joint angles.
3. Call your function with `measured`, the reset measurement prepared above. Inspect the result.
4. Check that the result has shape `(18,)`, a floating-point dtype, and only zero values.

**Done when:** the function returns the expected action and you can explain why it has 18 entries. Share the function and its check output for review. We will then connect your policy to a recorded rollout.


In [ ]:
def policy(observation):
    offsets = np.zeros(18, dtype=float)
    offsets[0] = 1
    offsets[1] = -1

    measured = observation[0]
    desired = observation[1]

    if measured > desired:
        offsets[0] = -1
    elif measured < desired:
        offsets[1] = -1
    return offsets


## Visual feedback and measurements

Work in this notebook. `simulate.py` runs the fixed SPAWN baseline; it does not call your policy.

Your draft above remains yours to finish. `observation` is a `MeasuredState` with named fields, not a two-item sequence. Choose a measurement and a separate desired value. Decide which actuator responds and how its offset changes on each side of the target. Return 18 finite offsets in radians.

The setup below supplies recording, replay, and plots. It does not implement the policy or training. A recording starts from a fresh robot reset. Each action uses the previous measurement and is held for `physics_steps` physics steps. The recording contains the reset and each action endpoint. Very short events between those samples are not shown.

Choose action timing and inspection length before running. Duration is `action_count * physics_steps * sim.model.opt.timestep`. This fixed inspection length is not an RL episode termination rule.

In [ ]:
from dataclasses import fields
from policy_feedback import record_policy, plot_recordings

print("Available measurements:", [field.name for field in fields(measured)])
print("Physics timestep (s):", sim.model.opt.timestep)
print("Actuator order:")
for index in range(sim.model.nu):
    print(index, sim.model.actuator(index).name)


### Check your action

After editing the policy cell, run it again to replace the function in the kernel. This check calls your policy once without stepping physics. A stateful policy may change its own internal state; reset that state before a comparison.

In [ ]:
action = np.asarray(policy(measured), dtype=float)
assert action.shape == (sim.model.nu,)
assert np.isfinite(action).all()
print(action)


### Record, watch, measure

Set `physics_steps` and `action_count` to your chosen positive integers. Then uncomment the calls below. Use a treatment label that includes your changed parameter values. These calls execute your policy when you run them.

Keep the previous recording under a different variable name when you change the policy. The replay never reevaluates the policy. Close each viewer window when finished.

The plots show world +X displacement, height, contacts, and one joint's measured angle against its clipped absolute target. These are inspection measurements, not a reward definition. `recording.offsets` contains requested offsets; `recording.targets` contains applied targets. Action times mark the start of each hold. Measurement times mark its end.

In [ ]:
# physics_steps = ...  # Physics steps per policy call: your timing choice.
# action_count = ...   # Number of policy calls in this inspection.
# recording = record_policy(
#     policy, label="draft: describe changed values here",
#     physics_steps=physics_steps, action_count=action_count,
# )
# viewer = recording.watch(REPO_ROOT / "telemetry" / "policy-replays", speed=1.0)
# fig, axes = plot_recordings(recording, actuator=0)


### Compare one change

Record your all-zero policy under the same timing and inspection length as a neutral-target control. It tells us what happens without your added offsets. It is separate from the support-aware STAND controller.

Watch both recordings before interpreting a difference. Compare with `plot_recordings(control, recording, actuator=...)`. Replay speed changes only display timing.

Keep your prediction, observed difference, and next change here. A single short inspection does not establish learned walking.